<a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

# Rank-One Model Editing (ROME)
This notebook enables interactive experimentation with ROME and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
if os.path.basename(os.getcwd()) == "KE4MHQ":
    os.chdir("rome")
!ls


import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution
import json
import time

from util.eval_greedy import eval_editing


import numpy as np
import random

def set_seed(seed=42):
    random.seed(seed)  # Python random module
    np.random.seed(seed)  # NumPy
    torch.manual_seed(seed)  # PyTorch CPU
    torch.cuda.manual_seed(seed)  # PyTorch GPU
    torch.cuda.manual_seed_all(seed)  # Multi-GPU
    torch.backends.cudnn.deterministic = True  # Ensure deterministic behavior
    torch.backends.cudnn.benchmark = False  # Disable auto-optimization

set_seed(42)

 baselines		      'hop1-Eval-[15]'		    hparams
'both-Eval-[5]-[10]'	      'hop1-Eval-[20]'		    LICENSE
'both-Eval-[5, 15]-[10, 20]'  'hop1-Eval-[5]'		    logs
'both-Eval-[5]-[5]'	      'hop1-Eval-[5, 10, 15, 20]'   notebooks
 CITATION.cff		       Hop1-Eval-5-10-15-20	    README.md
 data			      'hop2-Eval-[10]'		    results
 dsets			      'hop2-Eval-[15]'		    rome
 experiments		      'hop2-Eval-[20]'		    scripts
 globals.yml		      'hop2-Eval-[5]'		    util
'hop1-Eval-[10]'	      'hop2-Eval-[5, 10, 15, 20]'
 Hop1-Eval-10		       hop2-Eval-5-10-15-20


Here, you can specify a GPT model (`MODEL_NAME`).

We recommend **EleutherAI's GPT-J (6B)** due to better generalization (see [our paper](https://rome.baulab.info/) for details), but GPT-2 XL (1.5B) consumes less memory.
* `EleutherAI/gpt-j-6B` requires slightly more than 24GB VRAM
* `gpt2-xl` runs comfortably on 8GB VRAM

In [ ]:
IS_COLAB = False
ALL_DEPS = False
try:
    import google.colab, torch, os

    IS_COLAB = True
    os.chdir("/content/rome")
    if not torch.cuda.is_available():
        raise Exception("Change runtime type to include a GPU.")
except ModuleNotFoundError as _:
    pass


# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = 'cpu'
device = torch.device('cuda:3')
print(f"Using device: {device}")


Using device: cuda:3


In [4]:

# del model  # Deletes the model from memory
# torch.cuda.empty_cache()  # Clears unused memory from the GPU
# torch.cuda.ipc_collect()  # Helps reclaim unused memory


In [5]:
def save_mlp_layer(model, layer_idx, file_path):
    mlp_weights = model.transformer.h[layer_idx].mlp.state_dict()
    torch.save(mlp_weights, file_path)
    print(f"MLP layer {layer_idx} saved to {file_path}")

def load_mlp_layer(model, layer_idx, file_path):
    mlp_weights = torch.load(file_path)
    model.transformer.h[layer_idx].mlp.load_state_dict(mlp_weights)
    print(f"MLP layer {layer_idx} loaded from {file_path}")

In [11]:
ALG_NAME = "ROME-Multi" # "ROME-Multi"
MODEL_NAME = "EleutherAI/gpt-j-6B" # "gpt2-xl"
layers_to_edit = [10]

json_path_dict = {
    "gpt2-xl":"hparams/ROME/gpt2-xl.json",
    "EleutherAI/gpt-j-6B":"hparams/ROME/EleutherAI_gpt-j-6B.json"
}

with open(json_path_dict[MODEL_NAME], "r") as f:
    data = json.load(f)
data["layers"] = layers_to_edit
with open(json_path_dict[MODEL_NAME], "w") as f:
    json.dump(data, f, indent=2)



In [7]:

model, tok = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=IS_COLAB).to(
        device
    ),
    
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token = tok.eos_token
model.config

Some weights of the model checkpoint at EleutherAI/gpt-j-6B were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transformer.h.20.attn.bi

GPTJConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "EleutherAI/gpt-j-6B",
  "activation_function": "gelu_new",
  "architectures": [
    "GPTJForCausalLM"
  ],
  "attn_pdrop": 0.0,
  "bos_token_id": 50256,
  "embd_pdrop": 0.0,
  "eos_token_id": 50256,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gptj",
  "n_embd": 4096,
  "n_head": 16,
  "n_inner": null,
  "n_layer": 28,
  "n_positions": 2048,
  "resid_pdrop": 0.0,
  "rotary": true,
  "rotary_dim": 64,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50,
      "temperature": 1.0
    }
  },
  "tie_word_embeddings": false,
  "tokenizer_class": "GPT2Tokenizer",
  "torch_dtype": "float32",
  "transformers_version": "4.49.0",

#  Pipeline for testing multiple insertions on MQuake

In [8]:

# ds_file = "dsets/single_edit_0-100.json"
ds_file = "dsets/ds_classification/hop1_edits.json"
with open(ds_file, "r") as f:
    mhq_ds = json.load(f)

# context used when testing the editted model
context_file = "dsets/rel-prompts.json"
with open(context_file, "r") as f:
    rel_prompts = json.load(f)


print("Start time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
print("layers_to_edit: ", layers_to_edit)
print("ds_file: ", ds_file)

total = 100
correct = 0
# for i in range(len(mhq_ds)):
for i in range(total):
    case = mhq_ds[i]
    request = case["requested_rewrite"]
    generation_prompts = []

    print("\n\n"+4*"***********************************************")
    print(f"Request {i+1}, case_id: {case['case_id']}")


    # Restore fresh copy of model
    try:
        with torch.no_grad():
            for k, v in orig_weights.items():
                nethook.get_parameter(model, k)[...] = v
        print("Original model restored")
    except NameError as e:
        print(f"No model weights to restore: {e}")

    # Execute rewrite
    model_new, orig_weights = demo_model_editing(
        model, tok, request, generation_prompts, alg_name=ALG_NAME,generate_prompts=False 
        )

    if eval_editing(model, case, rel_prompts, tok, save_dir="Hop1-Eval-" + "-".join(map(str, layers_to_edit))):
        correct += 1

print(f"Correct: {correct}/{total}")
    

Start time:  2025-02-28 05:20:48
layers_to_edit:  [5, 10, 15, 20]
ds_file:  dsets/ds_classification/hop1_edits.json


********************************************************************************************************************************************************************************************
Request 1, case_id: 2
No model weights to restore: name 'orig_weights' is not defined

###########################################
#                                         #
#  Retrieving ROME-Multi hyperparameters  #
#                                         #
###########################################
Loading from hparams/ROME/EleutherAI_gpt-j-6B.json
ROMEHyperParams(layers=[5, 10, 15, 20], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=27, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.fc_out', layer_module_tmp='transformer.h.{}', 

We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class (https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)


Cached context templates ['{}', 'Q: . {}', 'Q: . {}', 'The present invention relates. {}', 'The role of the. {}', '\n \n-. {}', 'A new report from. {}', 'Q: . {}', 'Q: . {}', '\n \n=. {}', 'Q: . {}', 'The present invention relates to a method for producing. {}', ' Ask HN: Is there any. {}', " Show HN: I'm looking. {}", 'Q: Is there a way to. {}', ' Show HN: The best way. {}', 'Q: How to make a list. {}', 'Q: How to use a function. {}', 'Q: How do you get the. {}', 'Q: Can I get the current. {}', 'Q: What is a good way. {}']
Computing left vector (u)...
Selected u projection object Aslan
Retrieving inverse covariance statistics for EleutherAI_gpt-j-6B @ transformer.h.5.mlp.fc_out. The result will be cached to avoid repetitive computation.
Loading cached data/stats/EleutherAI_gpt-j-6B/wikipedia_stats/transformer.h.5.mlp.fc_out_float32_mom2_100000.npz


  0%|          | 0/1000 [00:00<?, ?it/s]

Left vector shape: torch.Size([16384])
Computing right vector (v)
Lookup index found: 1 | Sentence: Aslan was created by Charles Stur | Token: lan
Rewrite layer is 5
Tying optimization objective to 27
Recording initial value of v*
loss 5.145 = 5.145 + 0.0 + 0.0 avg prob of [ Charles Sturridge] 0.006765487603843212
loss 2.176 = 2.114 + 0.039 + 0.023 avg prob of [ Charles Sturridge] 0.1246512159705162
loss 0.671 = 0.581 + 0.054 + 0.037 avg prob of [ Charles Sturridge] 0.5634833574295044
loss 0.136 = 0.023 + 0.064 + 0.048 avg prob of [ Charles Sturridge] 0.9768180847167969
loss 0.143 = 0.016 + 0.069 + 0.058 avg prob of [ Charles Sturridge] 0.9842926263809204
loss 0.149 = 0.013 + 0.069 + 0.067 avg prob of [ Charles Sturridge] 0.987026572227478
loss 0.152 = 0.011 + 0.066 + 0.075 avg prob of [ Charles Sturridge] 0.989199161529541
loss 0.146 = 0.009 + 0.061 + 0.076 avg prob of [ Charles Sturridge] 0.99130779504776
loss 0.139 = 0.007 + 0.056 + 0.076 avg prob of [ Charles Sturridge] 0.992825329

...

  0%|          | 0/1000 [00:00<?, ?it/s]

Left vector shape: torch.Size([16384])
Computing right vector (v)
Lookup index found: 1 | Sentence: Aslan was created by Charles Stur | Token: lan
Rewrite layer is 20
Tying optimization objective to 27
Recording initial value of v*
loss 5.145 = 5.145 + 0.0 + 0.0 avg prob of [ Charles Sturridge] 0.006765487603843212
loss 4.34 = 4.317 + 0.007 + 0.017 avg prob of [ Charles Sturridge] 0.0165974423289299
loss 2.612 = 2.555 + 0.03 + 0.027 avg prob of [ Charles Sturridge] 0.10831264406442642
loss 0.775 = 0.694 + 0.044 + 0.037 avg prob of [ Charles Sturridge] 0.5574809312820435
loss 0.197 = 0.102 + 0.049 + 0.046 avg prob of [ Charles Sturridge] 0.9106481075286865
loss 0.122 = 0.015 + 0.052 + 0.055 avg prob of [ Charles Sturridge] 0.9855029582977295
loss 0.118 = 0.004 + 0.05 + 0.063 avg prob of [ Charles Sturridge] 0.9959802627563477
loss 0.11 = 0.003 + 0.042 + 0.065 avg prob of [ Charles Sturridge] 0.9973911046981812
loss 0.106 = 0.002 + 0.038 + 0.065 avg prob of [ Charles Sturridge] 0.9976009

/home/jeffhe/anaconda3/envs/KE4MHQ_env/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:677: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Evaluation result saved to:  Hop1-Eval-5-10-15-20/j-6B_id_2.json


********************************************************************************************************************************************************************************************
Request 2, case_id: 4
Original model restored

###########################################
#                                         #
#  Retrieving ROME-Multi hyperparameters  #
#                                         #
###########################################
Loading from hparams/ROME/EleutherAI_gpt-j-6B.json
ROMEHyperParams(layers=[5, 10, 15, 20], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=27, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.fc_out', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_module_tmp='transformer.h.{}.attn', ln_f_module

KeyError: 'hop'

In [20]:
import os
import json

# folder_dir = "hop2-Eval-5-10-15-20"
folder_dir ="both-Eval-[5, 15]-[10, 20]"
folder_dir = "both-Eval-[5]-[5]"
# folder_dir = "hop2-Eval-[10]"
folder_dir = "hop2-Eval-[20]"
folder_dir = "hop1-Eval-[5, 10, 15, 20]"
# folder_dir = "hop2-Eval-5-10-15-20"
correct = 0
total = len(os.listdir(folder_dir))
# iterate over the json files in the folder
for filename in os.listdir(folder_dir):
    if filename.endswith(".json"):
        with open(os.path.join(folder_dir, filename), "r") as f:
            data = json.load(f)
            if data[-1]["correct"]:
                correct += 1
print(f"{folder_dir} Correctness: {correct}/{total}")

hop1-Eval-[5, 10, 15, 20] Correctness: 65/240


In [17]:
! python3 -m experiments.summarize --dir_name=ROME-Multi --runs=run_003

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/jeffhe/KE4MHQ/KE4MHQ/rome/experiments/summarize.py", line 9, in <module>
    from util.globals import *
  File "/home/jeffhe/KE4MHQ/KE4MHQ/rome/util/__init__.py", line 1, in <module>
    from .logit_lens import LogitLens
  File "/home/jeffhe/KE4MHQ/KE4MHQ/rome/util/logit_lens.py", line 4, in <module>
    import torch
ModuleNotFoundError: No module named 'torch'
